# M2: DNS-BERT + Isolation Forest on CIC (Colab)
Owner: Thierry · Backup: Yehan. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
GPU runs go through the team Colab Pro (see the Trello card "[Neo] Team Colab Pro GPU runs").

**Before running, set two folders in the first cell:**
- `RAW_DIR`: Drive folder with the 5 CIC **PCAP** zips: `Benign.zip` (top level), `lightBenign.zip`, `lightAttacks.zip`,
  `heavyBenign.zip`, `heavyAttacks.zip`. Names like `Benign (1).zip` are fine.
- `SHARED`: the team's shared Drive folder for checkpoints and results. If it was shared with you, add a shortcut
  to it in My Drive first (right-click → Organize → Add shortcut).

The notebook builds the CIC query table, checks it against the contract (`contracts.py`), trains, scores, and saves
checkpoints + results to `SHARED`. A disconnect resumes from the last checkpoint: just Run all again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RAW_DIR = '/content/drive/MyDrive/ICT3214 datasets'   # the 5 CIC PCAP zips
SHARED  = '/content/drive/MyDrive/leakmate'           # team shared folder: checkpoints + results
CKPT    = 'v2'   # checkpoint folder suffix; change it to retrain from scratch instead of resuming
BRANCH  = 'main'
!mkdir -p "$SHARED/results"

In [ ]:
%cd /content
# Private repo: add a GitHub token in Colab Secrets (key icon) named GH_TOKEN, notebook access ON.
from google.colab import userdata
try:
    tok = userdata.get('GH_TOKEN') + '@'
except Exception:
    tok = ''   # public repo / no secret
!rm -rf leakmate && git clone -q -b $BRANCH https://{tok}github.com/TerminallyStoned/leakmate.git
%cd /content/leakmate
!pip -q install -r requirements-m2.txt

In [ ]:
# Tranco top-1M (benign names, ref [18]); cached in SHARED
import os
if not os.path.exists(f'{SHARED}/tranco.csv'):
    !wget -q -O /tmp/tranco.zip https://tranco-list.eu/top-1m.csv.zip && unzip -p /tmp/tranco.zip > "$SHARED/tranco.csv"
!mkdir -p data/external && cp "$SHARED/tranco.csv" data/external/tranco.csv && head -3 data/external/tranco.csv

In [ ]:
# Build the CIC query table from the zips (always rebuilt: about 1 min) and check it against the contract
import os, glob, re, zipfile
CIC = 'data/processed/cic/queries.parquet'

def target(fname):
    n = re.sub(r'[^a-z]', '', os.path.basename(fname).lower().replace('.zip', ''))
    for key, sub in [('lightattack', 'light_attacks'), ('heavyattack', 'heavy_attacks'),
                     ('lightbenign', 'light_benign'), ('heavybenign', 'heavy_benign'), ('benign', 'benign_main')]:
        if n.startswith(key):
            return sub
    return None

found = {z: target(z) for z in glob.glob(f'{RAW_DIR}/*.zip')}
print(found)
assert sorted(v for v in found.values() if v) == sorted(
    ['benign_main', 'light_benign', 'light_attacks', 'heavy_benign', 'heavy_attacks']), 'missing or extra zips'
for z, sub in found.items():
    dest = f'data/raw/cic/{sub}'
    zipfile.ZipFile(z).extractall(dest)
    for inner in glob.glob(f'{dest}/*.zip'):   # in case a zip holds another zip
        zipfile.ZipFile(inner).extractall(dest)
!python scripts/cic_to_queries.py --raw data/raw/cic --out $CIC
import pandas as pd
from contracts import validate
df = pd.read_parquet(CIC)
validate(df)   # stops here with a fix-it message if the table breaks docs/contracts.md
print('contract ok')
DATA, EXP = CIC, 'E2'
print(df.shape); print(df.groupby(['split', 'label']).size())

## 1. DNS-BERT, BPE tokenizer (main model)

In [ ]:
!python -m models.m2_run --model m2 --exp $EXP --data $DATA \
    --tranco data/external/tranco.csv --tranco-n 200000 \
    --tokenizer bpe --epochs 3 --batch-size 512 \
    --ckpt-dir "$SHARED/ckpt_bpe_$CKPT" --save-model models/weights/m2_bpe

## 2. Ablation 1: character tokenizer

In [ ]:
!python -m models.m2_run --model m2 --exp $EXP --tag ablation1_char --data $DATA \
    --tranco data/external/tranco.csv --tranco-n 200000 \
    --tokenizer char --epochs 3 --batch-size 512 \
    --ckpt-dir "$SHARED/ckpt_char_$CKPT" --save-model models/weights/m2_char

## 3. Isolation Forest baseline (benign-only)

In [ ]:
!python -m models.m2_run --model iforest --exp $EXP --data $DATA --save-model models/weights/iforest.joblib

## 4. Check the results against the contract (official metrics come from the shared eval module)

In [ ]:
!python contracts.py results/m2/E2.json results/m2/E2_ablation1_char.json results/iforest/E2.json
import json, glob, numpy as np
from sklearn.metrics import roc_auc_score
for f in ['results/m2/E2.json', 'results/m2/E2_ablation1_char.json', 'results/iforest/E2.json']:
    r = json.load(open(f)); y, s = np.array(r['labels']), np.array(r['scores'])
    print(f"{r['model']:22s} {r['experiment']:4s} AUC={roc_auc_score(y, s):.5f} threshold={r['threshold']:.4f} "
          f"train={r['train_time_s']:.0f}s lat={r['latency_ms_per_query']:.2f}ms")

In [ ]:
# copy results + weights to the shared folder; results/ is what goes into the PR
!cp -r results/m2 results/iforest "$SHARED/results/" && cp -r models/weights "$SHARED/" && ls -R "$SHARED/results" | head